In [1]:
import os
from dotenv import load_dotenv
from langchain_classic.retrievers import EnsembleRetriever
from langchain_ollama import ChatOllama, OllamaEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_community.vectorstores import FAISS
from langchain_community.retrievers import BM25Retriever
from langchain.chat_models.base import init_chat_model
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate, PromptTemplate

load_dotenv()

True

In [2]:
# Ollama:
llm = ChatOllama(model="gemma3:12b")
embedding = OllamaEmbeddings(model="qwen3-embedding:8b")

In [3]:
from langchain_community.document_loaders import Docx2txtLoader

try:
    docx_loader = Docx2txtLoader("../DataIngestParsing/data/docx/terms_and_condition_young_app.docx")
    docx = docx_loader.load()

    print(f"Processed into {len(docx)} documents")
    print(f"Content {docx[0].page_content[:50]}...")
    print(f"Metadata {docx[0].metadata}")
except Exception as e:
    print(f"Error processing {e}")

Processed into 1 documents
Content תנאי שימוש

כללי:

1. מבוטח יקר, אנו שמחים שבחרת ל...
Metadata {'source': '../DataIngestParsing/data/docx/terms_and_condition_young_app.docx'}


In [4]:
import re

raw_text = docx[0].page_content
source = docx[0].metadata["source"]

# Split only on top-level numbered paragraphs (1. 2. 3.)
# NOT on sub-paragraphs like 3.1 3.2 — the (?!\d) prevents that
parts = re.split(r'(?=\n\d+\.(?!\d))', raw_text)

chunks = []
for part in parts:
    text = part.strip()
    if text:
        chunks.append(Document(page_content=text, metadata={"source": source}))

print(f"Created {len(chunks)} chunks")
for i, chunk in enumerate(chunks):
    print(f"\n--- Chunk {i+1} ---")
    print(chunk.page_content[:150])


Created 65 chunks

--- Chunk 1 ---
תנאי שימוש

כללי:

--- Chunk 2 ---
1. מבוטח יקר, אנו שמחים שבחרת לעשות שימוש באפליקציית הרכב של תכנית הביטוח ״הפניקס צעיר״ (להלן: ״האפליקציה״, ו- "הפניקס" או "החברה") אשר דרכה ניתן לרכו

--- Chunk 3 ---
2. עם כניסתך לאפליקציה ובטרם תבצע פעולה כלשהי באמצעות האפליקציה, שימוש בשירות או במידע כלשהו הקיים באפליקציה, הנך מתבקש לקרוא בעיון ולאשר את תנאי השימ

--- Chunk 4 ---
3. תנאי השימוש ומדיניות הפרטיות להלן באים להוסיף על תנאי השימוש ומדיניות הפרטיות של אתר הפניקס
https://www.fnx.co.il/customer-service/terms-of-use/pri

--- Chunk 5 ---
4. ייתכן אפשרות שבמידע המוצג למבוטח באמצעות האפליקציה נפלו שיבושים ו/או שגיאות ו/או אי דיוקים. בכל מקרה של סתירה בין המידע שיתקבל במסגרת השירות לבין ה

--- Chunk 6 ---
5. הרישום והכניסה לאפליקציה, לרבות לרשימת הדיוור ו/או דרכי יצירת הקשר עם החברה וכן השימוש במידע שמסר המבוטח לחברה ו/או שהצטבר אודותיו בעת השימוש באפלי

--- Chunk 7 ---
6. נבקש לעדכנך כי בעת השימוש באפליקציה, עשויות להופיע פרסומות ומודעות המותאמות לך אישית, כ

In [5]:
# FAISS is an alternative simple vector store (memory only)
# dense_vectorstore = FAISS.from_documents(chunks, embedding=embedding)

dense_vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embedding,
    persist_directory="./vector_store",
    collection_name="terms_conditions"
)

# print(f"Added {len(chunks)} chunks to vector store")
# print(f"Total documents in store: {vector_store._collection.count()}")


In [12]:
# Denes Retriever - vector/semantic search: understands meaning and paraphrases. "Does Phoenix share my data?" → finds chunk about third-party even if exact words differ.
dense_retriever = dense_vectorstore.as_retriever()

# Sparse Retriever (BM25) - keyword frequency (classic IR): rewards exact term overlap. If the query contains "third-party", BM25 scores chunks containing that exact phrase higher.
sparse_retriever = BM25Retriever.from_documents(chunks)
sparse_retriever.k = 4

In [13]:
# The Combination of both give us the best results for embedding long text (unlike xlsx files)
hybrid_retriever = EnsembleRetriever(
    retrievers=[dense_retriever, sparse_retriever],
    weights=[0.7, 0.3]
)

In [14]:
# Similarity TEST:

query = "הפניקס הולכת לשתף צדדים שלישיים עם המידע שנאסף מהאפליקציה?"

# similar_docs = vector_store.similarity_search(query, k=2)
similar_docs_with_scores = dense_vectorstore.similarity_search_with_score(query, k=2)

# scores with ChromaDB:
# Closer to 0 = Similar result
# Zero means identical vectors

# print(similar_docs[0])
# print("---------")
print(similar_docs_with_scores[0])

(Document(metadata={'source': '../DataIngestParsing/data/docx/terms_and_condition_young_app.docx'}, page_content='28. הפניקס לא תשתף צדדים שלישיים, למעט איתוראן לשם תפעול שוטף של תכנית הביטוח, במידע המאפשר זיהוי אישי, אלא אם קיימת חובה לפי דין או אם הדבר מתבקש באופן מפורש על ידך או באופן אחר כמפורט בתנאי השימוש.'), 0.45322519540786743)


In [15]:
from langchain_core.prompts import ChatPromptTemplate

custom_prompt = ChatPromptTemplate.from_messages([
    ("system", """
        You are a professional customer service representative for Phoenix Insurance Company.
        ***Always respond in Hebrew regardless of the language used in the question***.

        Your goal is to provide accurate information about the company's terms and condition.

        Guidelines:
        * Use a professional, patient, and respectful tone.
        * If information is missing or not found in the system — state it clearly.
        * Do not fabricate information that does not exist in the provided data.
        * If the question is unrelated to terms and condition data — politely respond that you cannot assist with that topic.
        * Phrase answers in a natural, human way — not as a raw technical data list.

        ***Always respond in Hebrew***.

        Context: {context}
    """),
    ("human", "{input}")
])

custom_prompt

ChatPromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, template="\n        You are a professional customer service representative for Phoenix Insurance Company.\n        ***Always respond in Hebrew regardless of the language used in the question***.\n\n        Your goal is to provide accurate information about the company's terms and condition.\n\n        Guidelines:\n        * Use a professional, patient, and respectful tone.\n        * If information is missing or not found in the system — state it clearly.\n        * Do not fabricate information that does not exist in the provided data.\n        * If the question is unrelated to terms and condition data — politely respond that you cannot assist with that topic.\n        * Phrase answers in a natural, human way — not as a raw technical data list.\n\n        ***Always re

In [16]:
from langchain_classic.chains.retrieval import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain

document_chain = create_stuff_documents_chain(llm, custom_prompt)
rag_chain = create_retrieval_chain(hybrid_retriever, document_chain)


In [17]:
response = rag_chain.invoke({
    "input": "האם הפניקס יכולה להעביר את הפרטים שלי לצד שלישי?",
})

print(response.get("answer"))

שלום רב,

בהחלט, הפניקס עשויה להעביר את פרטיך לצדדים שלישיים, אך ישנם תנאים מוגדרים לכך. הפניקס רשאית להעביר את המידע שלך לצדדים שלישיים למטרות שונות, כגון:

*   בהתאם לבקשתך או בהסכמתך המפורשת.
*   לפי צו שיפוטי או דרישה של רשות מוסמכת.
*   במקרים של הפרת תנאי השימוש או מדיניות הפרטיות.
*   לגביית תשלומים המגיעים להפניקס.
*   למניעת הונאות.
*   במקרה בו העבירה הפניקס את פעילותה או זכויותיה לצד שלישי.

חשוב לציין, הפניקס לא תשתף צדדים שלישיים במידע המאפשר זיהוי אישי, אלא אם קיימת חובה לפי דין או אם הדבר מתבקש באופן מפורש ממך. כמו כן, הפניקס יכולה להעביר מידע אנונימי לכל גוף אחר.

אם יש לך שאלות נוספות בנוגע למדיניות הפרטיות של הפניקס, אשמח לעזור.
